# Prediction Thresholding

As an additional post-processing step for classification tasks that include an outside/other class ("O") label, we can require that predictions exceed some threshold of certainty that a sample (or token, in the case of token classification) is not a member of the outside class.<br>
<br>
For example, say we have 3 class labels standing for outside, person, location: O, PER, and LOC<br>

{
0: "O"
1: "PER"
2: "LOC"
}<br>
<br>
We have already trained our model and obtained our logits, and done a softmax:
[.45, .35, .2]<br>
Our prediction for the probability of "O" is .45, "PER" is .35, and "LOC" is .2.
<br>
Separately, we have also learned thresholds with which we can alter our predictions:<br>
<br>
[.5, .8]<br>
<br>
Each of these thresholds cooresponds to a non-"O" label in the problem. The threshold for "PER" is .5, and the threshold for "LOC" is .8.<br>
<br>
The process for applying these thresholds is as follows:<br>
For a prediction, find the highest probability for a non-"O" class.<br>
Check if that probability is lower or higher than the threshold value associated with that class<br>
If higher, predict the non-"O" class. If equal or lower, predict the "O" class.<br>
<br>
Let's walk through our example:

In [ ]:
# The class labels for our problem
label_dict = {0:'O', 1:'PER', 2:'LOC'}
# The thresholds we learned by some outside means
thresholds = [.5, .8]
# The softmaxed logits for a single token predicted by our NER model
logits = [.45, .35, .2]
# excluding "O" class
sub_logits = logits[1:]
# max index of sub_logits
max_idx = sub_logits.index(max(sub_logits))
# max index of original logits list (so long as we always keep "O" at index 0, a good practice)
max_idx+=1
# Applying threshold, thresholds index is 1 lower because of excluded "O" class
prediction = 0 # Predicting "O" by default here for simplicity
if logits[max_idx] > thresholds[max_idx - 1]:
    prediction = max_idx

print(f"Predicted class: {label_dict[prediction]}")


Why does this matter? Shouldn't we learn how to predict optimally during training already?
<br><br>
When we train our model, we pass back loss for incorrectly predicting a single token (usually, mistaking a token for a member of the "O" class or vice versa). We normally tune the model to maximize our macro f1 per token. <br>
 <br>
 Some methods of evaluation require capturing a full sequence of entities. For example, in a sample:<br>
 <br>
 "We went to the Siegel Center." <br>
 <br>
On a strict evaluation, the "Location" is "Siegel Center",a single entity at characters 15-27. If we were evaluating based on spans, the correct output is "15-27". If we classified only "Siegel" as a location, we would output "15-20". This would not evaluate as partially correct, it would evaluate as incorrect. <br>
<br>
The goal of including thresholding in our post-processing is to learn a set of parameters that helps with this type of strict evaluation that our model was not explicitly trained for.

# The Task:

For now, we don't have an implementation of our strict evaluation or our span-based output, so what is needed is a toy implementation of learning thresholds with **Optuna** which can later be refactored into the real post-processing. For the toy problem, you will: <br>
Read a pickled dataframe of samples, original predictions, and logits<br>
Find the original macro f1 obtained by using an argmax of the logits<br>
Use **Optuna** to learn a set of thresholds that maxizes macro f1<br>
Add a threshold prediction column to the dataframe using a method that applies your learned thresholds<br>
Print the macro f1 obtained by using the **Optuna** learned thresholds<br>
<br>
Sample code is provided for reading the dataframe and finding the macro f1 <br>
<br>
When working with dataframes, it can be helpful to find a way to view the data visually. In pycharm this is trivial, you can click "view as dataframe" on the dataframe variable while examining variables at a breakpoint while using the GDB style debugger to run a .py file, or while viewing the variables tab of a running a jupyter notebook. In VScode, an extension may be helpful.



In [ ]:
import pandas as pd
from sklearn.metrics import classification_report, f1_score

In [ ]:
# Load df, make structures for labels and preds to pass to skl methods
test_data = pd.read_pickle('../../outputs/test_preds/test_data.pkl')
y_true = test_data["Label"]
y_pred = test_data["Prediction"]
print(classification_report(y_true, y_pred))

In [ ]:
# This is a metric we would optimize over in the toy problem
f1_macro = f1_score(y_true, y_pred, average='macro')